In [ ]:
TRAINED_RESNEXT_MODEL_PATH = Path("./trained_pytorch_models/resnext50_32x4d.pth")
RESNEXT_RESULT_PATH = Path("./trained_pytorch_models/results_resnext50.csv")

if TRAINED_RESNEXT_MODEL_PATH.is_file() and RESNEXT_RESULT_PATH.is_file():
    print("Loading trained model...")

    model_resnext50 = models.resnext50_32x4d()
    model_resnext50.fc = nn.Linear(model_resnext50.fc.in_features, num_classes)

    state_dict = torch.load(
        TRAINED_RESNEXT_MODEL_PATH,
        map_location=device,
        weights_only=True
    )

    model_resnext50.load_state_dict(state_dict)
    model_resnext50.to(device)
    model_resnext50.eval()

    results_resnext50 = dp.read_csv(RESNEXT_RESULT_PATH)

else:
    print("Training new model...")

    model_resnext50 = models.resnext50_32x4d()
    model_resnext50.fc = nn.Linear(model_resnext50.fc.in_features, num_classes)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_resnext50.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=3
    )

    model_resnext50, results_resnext50 = train_evol_model(
        "ResNeXt50",
        model_resnext50,
        device,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        scheduler,
        50
    )

    TRAINED_RESNEXT_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

    torch.save(model_resnext50.cpu().state_dict(), TRAINED_RESNEXT_MODEL_PATH)
    model_resnext50.to(device)

    results_resnext50.to_csv(RESNEXT_RESULT_PATH, index=False)

    print(f"Model weights saved to {TRAINED_RESNEXT_MODEL_PATH}")
    print(f"Training results saved to {RESNEXT_RESULT_PATH}")